# Caracterización de la tirada en FATE2d6

Este notebook analiza matemáticamente la tirada base del sistema:

**2d6 + Atributo + Habilidad**

El objetivo es caracterizar cómo se comporta la tirada en términos de
probabilidad y dificultad, tomando como punto de partida una configuración
concreta del sistema definida en un archivo JSON.

## Objetivos del análisis

- Analizar la distribución de frecuencias del modificador `Atributo + Habilidad`.
- Calcular el valor esperado del modificador a la tirada (E[m]).
- Estudiar la distribución de probabilidad de distintas tiradas posibles.
- Comparar el comportamiento de la tirada frente a la escala de dificultades del sistema.
- Dejar una base reutilizable para analizar otras configuraciones del juego.

## Setup

En esta sección se prepara el entorno de trabajo del notebook:

- se resuelven las rutas del proyecto
- se importan las dependencias necesarias
- se carga la configuración del sistema desde un archivo JSON

Para reutilizar este notebook con otra variante del sistema, basta con cambiar
la ruta del archivo de configuración en la celda siguiente.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline

ROOT = Path().resolve().parent
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from fate2d6.config import load_config
from fate2d6.probability import (
    expected_modifier,
    modifier_distribution,
    modifier_probabilities,
    total_modifier_combinations,
    modifier_std,
    modifier_mode
)

CONFIG_PATH = ROOT / "config" / "base.json"
config = load_config(CONFIG_PATH)

print("LOAD OK")
print(f"Sistema cargado: {config.system_name}")
print(f"Configuración usada: {CONFIG_PATH}")

## Análisis de la Configuración

La configuración cargada define el conjunto de atributos, habilidades y valores
asociados que se usarán para calcular el espacio de modificadores posibles del
sistema.

Al cambiar el archivo JSON de configuración, este notebook puede reutilizarse
para analizar otras variantes de FATE2d6.

In [ ]:
print("Configuración:", config.system_name)
print("Atributos:", config.attribute_names)
print("Distribución de atributos:", config.attribute_value_distribution)
print("Habilidades:", config.skill_names)
print("Distribución de habilidades:", config.skill_value_distribution)

## Distribución del modificador `Atributo + Habilidad`

A continuación se calculan todas las combinaciones posibles entre los valores
de atributos y los valores de habilidades definidos en la configuración
cargada.

El objetivo de esta parte es obtener:

- la frecuencia de cada modificador posible
- la probabilidad asociada a cada uno
- el valor esperado del modificador

Esta distribución describe la estructura interna del sistema para la
configuración elegida. No representa todavía el uso real en mesa, ya que la
frecuencia efectiva de cada combinación dependerá también de la ficción y de
las decisiones de los jugadores.

In [ ]:
distribution = modifier_distribution(config)
probabilities = modifier_probabilities(config)
total = total_modifier_combinations(config)
expected = expected_modifier(config)
mode = modifier_mode(config)
std = modifier_std(config)

rows = []
for modifier, frequency in distribution.items():
    rows.append(
        {
            "Modificador": modifier,
            "Frecuencia": frequency,
            "Probabilidad": probabilities[modifier],
            "Porcentaje": probabilities[modifier] * 100,
        }
    )

df_modifiers = pd.DataFrame(rows)

df_modifiers_display = df_modifiers.copy()
df_modifiers_display["Probabilidad"] = df_modifiers_display["Probabilidad"].map(
    lambda x: f"{x:.3f}"
)
df_modifiers_display["Porcentaje"] = df_modifiers_display["Porcentaje"].map(
    lambda x: f"{x:.1f}%"
)

print("Configuración:", config.system_name)
print("Total de combinaciones:", total)
print("Valor esperado del modificador:", round(expected, 4))
print("Moda:", mode)
print("Desviación típica:", round(std, 4))

x = np.array([-2, -1, 0, 1, 2, 3, 4, 5])
y = np.array([0, 4, 12, 22, 26, 20, 6, 0])

x_smooth = np.linspace(-2, 5, 400)
spline = make_interp_spline(x, y, k=3)
y_smooth = spline(x_smooth)

plt.figure(figsize=(9, 5))

plt.bar(
    df_modifiers["Modificador"].values,
    df_modifiers["Frecuencia"].values,
    alpha=0.6,
    color="steelblue",
    label="Frecuencia",
)

plt.plot(
    x_smooth,
    y_smooth,
    color="darkorange",
    linewidth=2.5,
    label="Tendencia suavizada",
)

plt.axvline(
    expected,
    color="crimson",
    linestyle="--",
    linewidth=2,
    label=f"E[m] = {expected:.2f}",
)

plt.title("Distribución de frecuencias de Atributo + Habilidad")
plt.xlabel("Modificador")
plt.ylabel("Frecuencia")
plt.xticks(np.arange(-2, 6, 1))
plt.xlim(-2, 5)
plt.legend()
plt.grid(alpha=0.2)

plt.show()

df_modifiers_display

## Interpretación de la distribución de modificadores

La tabla y la gráfica anteriores muestran cómo se distribuyen los valores de
`Atributo + Habilidad` en función de la configuración cargada.

Esta distribución refleja:

- qué modificadores son más frecuentes dentro del sistema
- qué valores extremos aparecen con menor probabilidad
- dónde se sitúa el centro de gravedad del sistema (valor esperado)

En general, esta forma describe la estructura matemática del sistema para la
configuración seleccionada. No representa directamente el uso real en partida,
ya que:

- no todas las combinaciones atributo-habilidad son igualmente naturales
- la ficción restringe ciertas acciones
- los jugadores tenderán a optimizar sus elecciones

Por tanto, esta distribución debe interpretarse como una **base estructural**
sobre la que luego se superpone el comportamiento real en mesa.

---

### Caso particular: configuración base

Si se utiliza la configuración base del sistema, definida en `base.json`, los
valores observados son:

- `-1`: 4 casos  
- `0`: 12 casos  
- `1`: 22 casos  
- `2`: 26 casos  
- `3`: 20 casos  
- `4`: 6 casos  

Esto muestra una clara concentración en torno a los valores positivos,
especialmente `+2`, que es el modificador más frecuente.

El valor esperado del modificador es aproximadamente **1.71**, lo que indica que
el sistema, en términos estructurales, está desplazado hacia valores positivos.

Si se modifica la configuración JSON, esta distribución cambiará y deberá
interpretarse de nuevo siguiendo este mismo esquema.

## Valor esperado del modificador como baseline

Una forma útil de aproximar el comportamiento general del sistema es tomar el
valor esperado del modificador `Atributo + Habilidad` y usarlo como punto de
partida para analizar la tirada base.

Este enfoque no sustituye al análisis detallado de cada modificador posible,
pero sí proporciona una referencia global conservadora desde la que comparar
otras distribuciones más específicas.